In [ ]:
# ===============================================================
# 0) import
# ===============================================================
from pathlib import Path
import random, shutil, os, cv2, numpy as np, yaml

# ===============================================================
# 1) CONSTANTS
# ===============================================================
TARGET_CNT = 198                           # รูป/คลาส เท่า ๆ กัน
CLASS_MAPPING = {                          # class_id : ชื่อโฟลเดอร์
    0: 'BG',
    1: 'WSSV',
    2: 'WSSV_BG',
    3: 'Healthy'                          # NEW CLASS
}

# ▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼ แก้ไขตรงนี้ ▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼
ROOT       = Path('C:/Users/Admin/Desktop/กุ้ง/ShrimpDiseaseImageBD/Root') # <--- ใช้ Path() ครอบ
# ▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲

ANN_ROOT   = ROOT/'Annotated Diseased Shrimp Images'/'Annotated Diseased Shrimp Images'
RAW_ROOT   = ROOT/'Raw Images'/'Raw Images'
YOLO_DST   = Path('datasets/shrimp4cls')

# ===============================================================
# 2) DEBUG: ตรวจสอบโฟลเดอร์ก่อน
# ===============================================================
print("🔍 Checking folders...")
print(f"ROOT exists: {ROOT.exists()}")
print(f"ANN_ROOT exists: {ANN_ROOT.exists()}")
print(f"RAW_ROOT exists: {RAW_ROOT.exists()}")

# ตรวจสอบแต่ละโฟลเดอร์
for cid, folder in CLASS_MAPPING.items():
    if folder == 'Healthy':
        path = RAW_ROOT/'Healthy'
        print(f"Class {cid} ({folder}): {path.exists()} - {path}")
        if path.exists():
            files = list(path.glob('*.*'))
            print(f"  → Found {len(files)} files")
    else:
        img_path = ANN_ROOT/folder/'images'
        lbl_path = ANN_ROOT/folder/'labels'
        print(f"Class {cid} ({folder}):")
        print(f"  Images: {img_path.exists()} - {img_path}")
        print(f"  Labels: {lbl_path.exists()} - {lbl_path}")
        if img_path.exists():
            imgs = list(img_path.glob('*.*'))
            print(f"  → Found {len(imgs)} images")
        if lbl_path.exists():
            lbls = list(lbl_path.glob('*.txt'))
            print(f"  → Found {len(lbls)} labels")

# ===============================================================
# 3) รวมไฟล์ทั้งหมด + เตรียม label สำหรับ Healthy (with validation)
# ===============================================================
# ===============================================================
# 3) รวมไฟล์ทั้งหมด + เตรียม label สำหรับ Healthy (with validation)
# ===============================================================
def collect_items():
    items = []  # [{'img':Path,'lbl':Path|None,'cid':int}, …]
    print("\n🔍 Collecting and verifying all items...")
    for cid, folder in CLASS_MAPPING.items():
        if folder == 'Healthy':
            img_dir = RAW_ROOT/'1. Healthy'
            if not img_dir.exists():
                print(f"❌ Warning: {img_dir} not found!")
                continue

            for img_path in img_dir.glob('*.*'):
                if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']:
                    # --- ส่วนที่เพิ่มเข้ามาเพื่อตรวจสอบไฟล์ ---
                    img_check = cv2.imread(str(img_path))
                    if img_check is not None:
                        items.append({'img': img_path, 'lbl': None, 'cid': cid})
                    else:
                        print(f"⚠️ Skipped unreadable or corrupt image: {img_path.name}")
                    # ------------------------------------

        else:  # 3 คลาสที่มี label
            img_dir   = ANN_ROOT/folder/'images'
            label_dir = ANN_ROOT/folder/'labels'

            if not img_dir.exists() or not label_dir.exists():
                print(f"❌ Warning: image or label dir for {folder} not found!")
                continue

            for img in img_dir.glob('*.*'):
                if img.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']:
                    lbl = label_dir/(img.stem + '.txt')
                    if lbl.exists():
                        items.append({'img': img, 'lbl': lbl, 'cid': cid})
                    else:
                        print(f"⚠️ Label not found for: {img.name}")
    return items

all_items = collect_items()
print(f"\n📊 Total items collected: {len(all_items)}")

# ===============================================================
# 4) ตรวจสอบข้อมูลแต่ละคลาส
# ===============================================================
print("\n📋 Items per class:")
for cid in CLASS_MAPPING:
    cls_items = [x for x in all_items if x['cid'] == cid]
    print(f"  Class {cid} ({CLASS_MAPPING[cid]}): {len(cls_items)} items")
    
    # ถ้าไม่มีข้อมูลเลย ให้หยุด
    if len(cls_items) == 0:
        print(f"❌ ERROR: No data found for class {cid}!")
        print("Please check your folder structure and file paths.")
        exit()

# ===============================================================
# 5) Balance dataset → ทุกคลาส = TARGET_CNT (with validation)
# ===============================================================
print(f"\n⚖️ Balancing to {TARGET_CNT} items per class...")
balanced = []
for cid in CLASS_MAPPING:
    cls_items = [x for x in all_items if x['cid'] == cid]
    
    if len(cls_items) == 0:
        print(f"❌ ERROR: No items for class {cid}")
        continue
        
    if len(cls_items) >= TARGET_CNT:                               # undersample
        sampled = random.sample(cls_items, TARGET_CNT)
        balanced += sampled
        print(f"  Class {cid}: undersampled {len(cls_items)} → {TARGET_CNT}")
    else:                                                          # oversample
        extra_needed = TARGET_CNT - len(cls_items)
        extra = random.choices(cls_items, k=extra_needed)
        balanced += cls_items + extra
        print(f"  Class {cid}: oversampled {len(cls_items)} → {TARGET_CNT}")

random.shuffle(balanced)
print(f"✅ Balanced dataset: {len(balanced)} total items")

# ===============================================================
# 6) Split 7 : 2 : 1  (train / val / test)
# ===============================================================
n = len(balanced)
train_split = int(.7 * n)
val_split = int(.9 * n)

train = balanced[:train_split]
val = balanced[train_split:val_split]
test = balanced[val_split:]

print(f"\n📊 Data split:")
print(f"  Train: {len(train)} ({len(train)/n*100:.1f}%)")
print(f"  Val: {len(val)} ({len(val)/n*100:.1f}%)")
print(f"  Test: {len(test)} ({len(test)/n*100:.1f}%)")

# ===============================================================
# 7) Helper : copy + สร้าง/แก้ไข label
# ===============================================================
def ensure_dir(p): p.mkdir(parents=True, exist_ok=True)

def write_healthy_label(img_path, save_path):
    """สร้าง bbox เต็มภาพสำหรับ Healthy"""
    try:
        img = cv2.imread(str(img_path))
        if img is None:
            print(f"❌ Cannot read image: {img_path}")
            return
        h, w = img.shape[:2]
        # YOLO norm : (x_center, y_center, w, h) → 0-1
        line = f"3 0.5 0.5 1.0 1.0\n"
        with open(save_path, 'w') as f:
            f.write(line)
    except Exception as e:
        print(f"❌ Error writing healthy label for {img_path}: {e}")

def rewrite_and_save(lbl_src, cid_new, save_path):
    """คัดลอก label เดิม แต่เปลี่ยน class_id เป็น cid_new"""
    try:
        with open(lbl_src, 'r') as f:
            lines = f.readlines()
        new_lines = []
        for ln in lines:
            parts = ln.strip().split()
            if len(parts) >= 5:
                parts[0] = str(cid_new)       # เปลี่ยน class id
                new_lines.append(' '.join(parts) + '\n')
        with open(save_path, 'w') as f:
            f.writelines(new_lines)
    except Exception as e:
        print(f"❌ Error rewriting label {lbl_src}: {e}")

def copy_split(split_name, data_list):
    print(f"\n📁 Copying {split_name} split ({len(data_list)} items)...")
    img_dir = YOLO_DST/split_name/'images'
    lbl_dir = YOLO_DST/split_name/'labels'
    ensure_dir(img_dir); ensure_dir(lbl_dir)

    for idx, it in enumerate(data_list):
        try:
            new_img_name  = f"{split_name}_{idx:05d}{it['img'].suffix}"
            new_lbl_name  = f"{split_name}_{idx:05d}.txt"
            shutil.copy(it['img'], img_dir/new_img_name)

            lbl_path = lbl_dir/new_lbl_name
            if it['cid'] == 3:                                  # Healthy
                write_healthy_label(it['img'], lbl_path)
            else:                                               # 3 คลาสเดิม
                rewrite_and_save(it['lbl'], it['cid'], lbl_path)
        except Exception as e:
            print(f"❌ Error copying item {idx}: {e}")

copy_split('train', train)
copy_split('val',   val)
copy_split('test',  test)

print(f"\n✅ Dataset balanced + copied → {YOLO_DST}")

# ===============================================================
# 8) สร้าง data.yaml ใหม่ (4 classes)
# ===============================================================
data_yaml = {
    'path': str(YOLO_DST.resolve()),
    'train': 'train/images',
    'val':   'val/images',
    'test':  'test/images',
    'nc': 4,
    'names': ['BG', 'WSSV', 'WSSV_BG', 'Healthy']
}

yaml_path = YOLO_DST/'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)
print(f"✅ data.yaml created: {yaml_path}")

# ===============================================================
# 9) Final verification
# ===============================================================
print(f"\n🔍 Final verification:")
for split in ['train', 'val', 'test']:
    img_count = len(list((YOLO_DST/split/'images').glob('*.*')))
    lbl_count = len(list((YOLO_DST/split/'labels').glob('*.txt')))
    print(f"  {split}: {img_count} images, {lbl_count} labels {'✅' if img_count == lbl_count else '❌'}")

print(f"\n🎉 Ready for training with 4 classes and balanced dataset!")

🔍 Checking folders...
ROOT exists: True
ANN_ROOT exists: True
RAW_ROOT exists: True
Class 0 (BG):
  Images: True - C:\Users\Admin\Desktop\กุ้ง\ShrimpDiseaseImageBD\Root\Annotated Diseased Shrimp Images\Annotated Diseased Shrimp Images\BG\images
  Labels: True - C:\Users\Admin\Desktop\กุ้ง\ShrimpDiseaseImageBD\Root\Annotated Diseased Shrimp Images\Annotated Diseased Shrimp Images\BG\labels
  → Found 198 images
  → Found 198 labels
Class 1 (WSSV):
  Images: True - C:\Users\Admin\Desktop\กุ้ง\ShrimpDiseaseImageBD\Root\Annotated Diseased Shrimp Images\Annotated Diseased Shrimp Images\WSSV\images
  Labels: True - C:\Users\Admin\Desktop\กุ้ง\ShrimpDiseaseImageBD\Root\Annotated Diseased Shrimp Images\Annotated Diseased Shrimp Images\WSSV\labels
  → Found 328 images
  → Found 328 labels
Class 2 (WSSV_BG):
  Images: True - C:\Users\Admin\Desktop\กุ้ง\ShrimpDiseaseImageBD\Root\Annotated Diseased Shrimp Images\Annotated Diseased Shrimp Images\WSSV_BG\images
  Labels: True - C:\Users\Admin\Desktop

: 